<a href="https://colab.research.google.com/github/nicowilliamxvii/TQHDC_CS441/blob/main/Waffle_%26Word_Charts_thloc2202137.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
df = pd.read_excel('/content/drive/MyDrive/cacfilecsvbaitaptqhdl/Canada-1.xlsx',
                       sheet_name='Canada by Citizenship',
                       skiprows=range(20),
)
df.head()

In [ ]:
print(df.head())

In [ ]:
# clean up the dataset to remove unnecessary columns (eg. REG)
columns_to_drop = ['AREA','REG','DEV','Type','Coverage']
df.drop(columns=[col for col in columns_to_drop if col in df.columns], axis=1, inplace=True)

# let's rename the columns so that they make sense
df.rename (columns = {'OdName':'Country', 'AreaName':'Continent','RegName':'Region'}, inplace = True)

# for sake of consistency, let's also make all column labels of type string
df.columns = list(map(str, df.columns))

# years that we will be using in this lesson - useful for plotting later on
years = list(map(str, range(1980, 2014))) # Define years before usage

# add total column
df['Total'] =  df[years].sum (axis = 1)

# set the country name as index - useful for quickly looking up countries using .loc method
# Add a check to prevent KeyError if 'Country' is already the index or not a column
if 'Country' in df.columns:
    df.set_index('Country', inplace = True)

print ('data dimensions:', df.shape)

In [ ]:
%matplotlib inline

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches # needed for waffle Charts

mpl.style.use('ggplot') # optional: for ggplot-like style

# check for latest version of Matplotlib
print ('Matplotlib version: ', mpl.__version__) # >= 2.0.0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def create_waffle_chart(categories, values, height, width, colormap='coolwarm', value_sign='', title='Waffle Chart'):
    """
    Tạo biểu đồ waffle chart từ dữ liệu categories và values
    """
    # Step 1: Tính tỷ lệ của mỗi danh mục
    total_values = sum(values)
    category_proportions = [float(v) / total_values for v in values]

    # Step 2: Tính tổng số ô
    total_num_tiles = height * width

    # Step 3: Tính số ô cho mỗi danh mục
    tiles_per_category = [round(proportion * total_num_tiles) for proportion in category_proportions]

    # Điều chỉnh để đảm bảo tổng số ô khớp
    if sum(tiles_per_category) != total_num_tiles:
        diff = total_num_tiles - sum(tiles_per_category)
        idx_max = max(range(len(values)), key=lambda i: values[i])
        tiles_per_category[idx_max] += diff

    # Step 4: Tạo ma trận waffle
    waffle_chart = np.zeros((height, width))
    category_index = 0
    tile_index = 0

    for col in range(width):
        for row in range(height):
            tile_index += 1
            if tile_index > sum(tiles_per_category[0:category_index]):
                category_index += 1
            waffle_chart[row, col] = category_index

    # Step 5-7: Tạo và làm đẹp biểu đồ
    if isinstance(colormap, str):
        cmap = plt.colormaps[colormap]
    else:
        cmap = colormap

    fig = plt.figure(figsize=(width * 0.5, height * 0.5))
    plt.matshow(waffle_chart, cmap=cmap, fignum=0)
    plt.colorbar()

    # Làm đẹp biểu đồ
    ax = plt.gca()
    ax.set_xticks(np.arange(-0.5, width, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, height, 1), minor=True)
    ax.grid(which='minor', color='w', linestyle='-', linewidth=2)
    plt.xticks([])
    plt.yticks([])

    # Tạo legend
    values_cumsum = np.cumsum(values)
    legend_handles = []

    for i, category in enumerate(categories):
        label_str = f'{category}: {values[i]:,}{value_sign}'
        color_val = cmap(float(values_cumsum[i]) / total_values)
        legend_handles.append(mpatches.Patch(color=color_val, label=label_str))

    plt.legend(
        handles=legend_handles,
        loc='lower center',
        ncol=min(len(categories), 4),
        bbox_to_anchor=(0.5, -0.2),
        frameon=False
    )

    plt.title(title, fontsize=14, pad=20)
    plt.tight_layout()

    return fig

# ===================================================
# Đường dẫn file
# ===================================================
file_path = '/content/drive/MyDrive/cacfilecsvbaitaptqhdl/Canada-1.xlsx'

print("Đang đọc dữ liệu từ file Excel...")
print(f"Đường dẫn: {file_path}")

# ===================================================
# Đọc sheet "Regions by Citizenship" - Đọc thủ công
# ===================================================

# Đọc toàn bộ file không header
df_raw_regions = pd.read_excel(file_path, sheet_name='Regions by Citizenship', header=None)

# Tìm dòng bắt đầu dữ liệu (dòng có 'Immigrants' hoặc 'Classification')
start_row = None
for i in range(50):
    if df_raw_regions.iloc[i, 0] == 'Immigrants' or df_raw_regions.iloc[i, 0] == 'Classification':
        start_row = i
        break

print(f"\nDòng bắt đầu dữ liệu khu vực: {start_row}")

# Đọc lại với header đúng
df_regions = pd.read_excel(file_path, sheet_name='Regions by Citizenship', header=start_row)

print("\nCác cột sau khi đọc đúng header:")
print(df_regions.columns.tolist())

# Xác định các cột cần thiết
type_col = df_regions.columns[0]  # 'Classification' hoặc 'Type'
coverage_col = df_regions.columns[1]  # 'Coverage'
region_col = df_regions.columns[3]  # 'RegName'

# Lọc dữ liệu Immigrants và Foreigners
df_foreigners = df_regions[(df_regions[type_col] == 'Immigrants') &
                            (df_regions[coverage_col] == 'Foreigners')]

# Các khu vực chính
regions = ['Africa Total', 'Asia Total', 'Europe Total', 'Latin America and the Caribbean Total',
           'Northern America', 'Oceania Total']

# Lấy dữ liệu năm 2013 (cột cuối cùng)
year_2013_values = []
valid_regions = []

print("\n" + "="*60)
print("DỮ LIỆU NHẬP CƯ THEO KHU VỰC NĂM 2013")
print("="*60)

for region in regions:
    region_row = df_foreigners[df_foreigners[region_col] == region]
    if not region_row.empty:
        value = region_row.iloc[0, -1]  # Cột cuối là năm 2013
        if pd.notna(value):
            year_2013_values.append(value)
            valid_regions.append(region)
            print(f"{region:45s}: {value:>12,} người")

# Tổng số
if valid_regions:
    total_2013 = sum(year_2013_values)
    print(f"\n{'TỔNG CỘNG':45s}: {total_2013:>12,} người")

    # Tạo waffle chart
    fig1 = create_waffle_chart(
        categories=valid_regions,
        values=year_2013_values,
        height=12,
        width=18,
        colormap='tab20',
        value_sign=' người',
        title='Waffle Chart - Phân bố nhập cư theo khu vực (2013)'
    )
    plt.show()

# ===================================================
# Đọc sheet "Canada by Citizenship"
# ===================================================

print("\n" + "="*60)
print("ĐANG ĐỌC DỮ LIỆU QUỐC GIA...")
print("="*60)

# Đọc toàn bộ file không header
df_raw_countries = pd.read_excel(file_path, sheet_name='Canada by Citizenship', header=None)

# Tìm dòng bắt đầu dữ liệu
start_row_country = None
for i in range(50):
    if df_raw_countries.iloc[i, 0] == 'Immigrants' or df_raw_countries.iloc[i, 0] == 'Classification':
        start_row_country = i
        break

print(f"Dòng bắt đầu dữ liệu quốc gia: {start_row_country}")

# Đọc lại với header đúng
df_countries = pd.read_excel(file_path, sheet_name='Canada by Citizenship', header=start_row_country)

print(f"Shape: {df_countries.shape}")

# Xác định các cột
type_col = df_countries.columns[0]
coverage_col = df_countries.columns[1]
country_col = df_countries.columns[2]  # OdName hoặc tên quốc gia

print(f"Cột loại: {type_col}")
print(f"Cột coverage: {coverage_col}")
print(f"Cột quốc gia: {country_col}")

# Lọc dữ liệu
df_filtered = df_countries[(df_countries[type_col] == 'Immigrants') &
                            (df_countries[coverage_col] == 'Foreigners')]

# Loại bỏ dòng Unknown
df_filtered = df_filtered[df_filtered[country_col] != 'Unknown']
df_filtered = df_filtered[df_filtered[country_col] != 'Total']

# Lấy top 10 quốc gia theo năm 2013 (cột cuối cùng)
if not df_filtered.empty:
    year_col = df_filtered.columns[-1]  # Cột 2013
    df_sorted = df_filtered.sort_values(by=year_col, ascending=False)
    top_countries = df_sorted.head(10)[[country_col, year_col]]

    print("\nTOP 10 QUỐC GIA CÓ NGƯỜI NHẬP CƯ NHIỀU NHẤT NĂM 2013:")
    print("-" * 60)
    for idx, row in top_countries.iterrows():
        print(f"{row[country_col]:35s}: {row[year_col]:>15,} người")

    # Tạo waffle chart cho top 10
    fig3 = create_waffle_chart(
        categories=top_countries[country_col].tolist(),
        values=top_countries[year_col].tolist(),
        height=10,
        width=14,
        colormap='viridis',
        value_sign=' người',
        title='Top 10 Quốc Gia Nhập Cư Nhiều Nhất vào Canada (2013)'
    )
    plt.show()

# ===================================================
# Phân tích xu hướng thời gian
# ===================================================

print("\n" + "="*60)
print("PHÂN TÍCH XU HƯỚNG NHẬP CƯ THEO THỜI GIAN")
print("="*60)

try:
    if valid_regions:
        fig2, ax = plt.subplots(figsize=(14, 6))

        # Lấy danh sách các năm từ cột (bỏ qua các cột đầu)
        year_cols = [col for col in df_foreigners.columns if str(col).isdigit() and 1980 <= int(col) <= 2013]

        for region in valid_regions:
            region_row = df_foreigners[df_foreigners[region_col] == region]
            if not region_row.empty:
                values = []
                years = []
                for year in year_cols:
                    val = region_row.iloc[0][year]
                    if pd.notna(val):
                        values.append(val)
                        years.append(int(year))

                if values:
                    ax.plot(years, values, marker='o', linewidth=2, markersize=4, label=region)

        ax.set_xlabel('Năm', fontsize=12)
        ax.set_ylabel('Số lượng người nhập cư', fontsize=12)
        ax.set_title('Xu hướng nhập cư vào Canada theo khu vực (1980-2013)', fontsize=14)
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_xticks(range(1980, 2015, 5))
        plt.tight_layout()
        plt.show()
    else:
        print("Không có dữ liệu khu vực để vẽ biểu đồ xu hướng")

except Exception as e:
    print(f"Không thể vẽ biểu đồ xu hướng: {e}")

# ===================================================
# Thống kê chi tiết
# ===================================================

print("\n" + "="*60)
print("THỐNG KÊ TỔNG QUAN")
print("="*60)

if valid_regions:
    print(f"Tổng số người nhập cư vào Canada năm 2013: {total_2013:>15,} người")

    print("\nPhân bố theo khu vực:")
    print("-" * 70)
    for region, value in zip(valid_regions, year_2013_values):
        pct = (value / total_2013) * 100
        bar_length = int(pct / 2)
        bar = '█' * bar_length if bar_length > 0 else '░'
        print(f"{region:40s}: {value:>12,} người ({pct:>5.1f}%) {bar}")

print("\n✅ Hoàn thành!")

In [ ]:
# install wordcloud
!pip install wordcloud==1.4.1

# import package and its set of stopwords
from wordcloud import WordCloud, STOPWORDS

print ('Wordcloud is installed and imported!')

In [ ]:
# download file and save as alice_novel.txt
!wget --quiet https://ibm.box.com/shared/static/m54sjtrshpt5su20dzesl5en9xa5vfz1.txt -O alice_novel.txt

# open the file and read it into a variable alice_novel
alice_novel = open('alice_novel.txt', 'r').read()

print ('File downloaded and saved!')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS

# ===================================================
# Đường dẫn file
# ===================================================
file_path = '/content/drive/MyDrive/cacfilecsvbaitaptqhdl/Canada-1.xlsx'

print("Đang đọc dữ liệu từ file Excel...")
print("="*60)

# ===================================================
# Đọc và xử lý dữ liệu
# ===================================================

# Đọc sheet "Canada by Citizenship" với header tại dòng 19
df_can = pd.read_excel(file_path, sheet_name='Canada by Citizenship', header=19)

print(f"Shape của dữ liệu: {df_can.shape}")

# ===================================================
# Xác định các cột dữ liệu
# ===================================================

# Cột quốc gia
country_col = 'Origin/Destination'

# Cột loại và coverage
type_col = 'Classification'
coverage_col = 'Unnamed: 1'

# Lọc dữ liệu Immigrants và Foreigners
df_filtered = df_can[(df_can[type_col] == 'Immigrants') &
                      (df_can[coverage_col] == 'Foreigners')]

print(f"Số dòng sau khi lọc: {len(df_filtered)}")

# Tìm các cột năm (các cột từ Unnamed: 9 đến Unnamed: 42)
year_columns = []
for col in df_filtered.columns:
    if 'Unnamed:' in str(col):
        try:
            idx = int(str(col).split(':')[1].strip())
            if 9 <= idx <= 42:
                year_columns.append(col)
        except:
            pass

print(f"\nSố cột năm tìm thấy: {len(year_columns)}")

# ===================================================
# Tạo dữ liệu tổng hợp
# ===================================================

if len(df_filtered) > 0 and len(year_columns) > 0:
    # Đặt tên quốc gia làm index
    df_filtered.set_index(country_col, inplace=True)

    # Chuyển đổi dữ liệu sang số
    for col in year_columns:
        df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce')

    # Tính tổng số người nhập cư (1980-2013)
    df_filtered['Total'] = df_filtered[year_columns].sum(axis=1)

    # Loại bỏ NaN và 0
    df_filtered = df_filtered[df_filtered['Total'] > 0]
    df_filtered = df_filtered[~df_filtered.index.isin(['Unknown', 'Total', 'World'])]

    print(f"\n✅ Số quốc gia có dữ liệu: {len(df_filtered)}")
    total_immigration = df_filtered['Total'].sum()
    print(f"✅ Tổng số người nhập cư (1980-2013): {total_immigration:,.0f}")

    # Hiển thị top 10
    print("\n" + "="*60)
    print("TOP 10 QUỐC GIA NHẬP CƯ NHIỀU NHẤT")
    print("="*60)
    top10 = df_filtered.nlargest(10, 'Total')
    for i, (country, row) in enumerate(top10.iterrows(), 1):
        pct = (row['Total'] / total_immigration) * 100
        print(f"{i:2d}. {country[:35]:35s}: {row['Total']:>12,.0f} ({pct:>5.1f}%)")

    # ===================================================
    # TẠO WORDCLOUD
    # ===================================================

    immigration_dict = df_filtered['Total'].to_dict()

    # 1. WordCloud cơ bản
    print("\n" + "="*60)
    print("1. WORDCLOUD CƠ BẢN")
    print("="*60)

    wordcloud = WordCloud(
        width=1200,
        height=800,
        background_color='white',
        colormap='RdYlGn',
        max_words=100,
        relative_scaling=0.5,
        prefer_horizontal=0.7
    ).generate_from_frequencies(immigration_dict)

    plt.figure(figsize=(16, 10))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('NHẬP CƯ VÀO CANADA (1980-2013)\nKích thước chữ tỷ lệ với số lượng người nhập cư',
              fontsize=16, pad=20, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 2. WordCloud với mask hình trái tim
    print("\n" + "="*60)
    print("2. WORDCLOUD HÌNH TRÁI TIM")
    print("="*60)

    # Tạo mask hình trái tim
    size = 800
    heart_mask = np.zeros((size, size), dtype=np.uint8)

    for y in range(size):
        for x in range(size):
            x_norm = (x - size/2) / (size/2)
            y_norm = (y - size/2) / (size/2)
            x2 = x_norm * 1.5
            y2 = y_norm * 1.5
            equation = (x2**2 + y2**2 - 1)**3 - x2**2 * y2**3
            if equation <= 0:
                heart_mask[y, x] = 255

    wordcloud_heart = WordCloud(
        width=size,
        height=size,
        background_color='white',
        mask=heart_mask,
        colormap='Reds',
        max_words=100,
        relative_scaling=0.5,
        contour_width=2,
        contour_color='darkred'
    ).generate_from_frequencies(immigration_dict)

    plt.figure(figsize=(12, 12))
    plt.imshow(wordcloud_heart, interpolation='bilinear')
    plt.axis('off')
    plt.title('CANADA - ĐIỂM ĐẾN CỦA THẾ GIỚI\n(Trái tim nhập cư)',
              fontsize=16, pad=20, fontweight='bold', color='darkred')
    plt.tight_layout()
    plt.show()

    # 3. WordCloud từ text (top 50)
    print("\n" + "="*60)
    print("3. WORDCLOUD TỪ TEXT (TOP 50 QUỐC GIA)")
    print("="*60)

    word_string = ''
    for country, value in list(immigration_dict.items())[:50]:
        repeat = max(1, int((value / total_immigration) * 300))
        word_string += (country + ' ') * repeat

    wordcloud_text = WordCloud(
        width=1000,
        height=600,
        background_color='white',
        colormap='viridis',
        max_words=50
    ).generate(word_string)

    plt.figure(figsize=(14, 8))
    plt.imshow(wordcloud_text, interpolation='bilinear')
    plt.axis('off')
    plt.title('TOP 50 QUỐC GIA NHẬP CƯ NHIỀU NHẤT VÀO CANADA\n(1980-2013)',
              fontsize=16, pad=20, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 4. WordCloud hình lá phong
    print("\n" + "="*60)
    print("4. WORDCLOUD HÌNH LÁ PHONG")
    print("="*60)

    # Tạo mask hình lá phong
    size = 800
    maple_mask = np.zeros((size, size), dtype=np.uint8)
    center_x, center_y = size // 2, size // 2

    for y in range(size):
        for x in range(size):
            dx = (x - center_x) / (size/2)
            dy = (y - center_y) / (size/2)
            angle = np.arctan2(dy, dx)
            r = np.sqrt(dx**2 + dy**2)
            num_lobes = 5
            lobe_factor = 1 + 0.5 * np.cos(num_lobes * angle)
            if r < (0.8 * lobe_factor):
                maple_mask[y, x] = 255

    wordcloud_maple = WordCloud(
        width=size,
        height=size,
        background_color='white',
        mask=maple_mask,
        colormap='RdBu_r',
        max_words=100,
        relative_scaling=0.6,
        contour_width=2,
        contour_color='darkred'
    ).generate_from_frequencies(immigration_dict)

    plt.figure(figsize=(12, 12))
    plt.imshow(wordcloud_maple, interpolation='bilinear')
    plt.axis('off')
    plt.title('NHẬP CƯ VÀO CANADA\n(Biểu tượng lá phong)',
              fontsize=16, pad=20, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 5. WordCloud với màu sắc tùy chỉnh
    print("\n" + "="*60)
    print("5. WORDCLOUD MÀU SẮC TÙY CHỈNH")
    print("="*60)

    # Hàm tạo màu tùy chỉnh
    top10_countries = list(top10.index)

    def color_func(word, font_size, position, orientation, random_state=None, **kwargs):
        if word in top10_countries[:3]:
            return "#E63946"
        elif word in top10_countries[3:7]:
            return "#457B9D"
        elif word in top10_countries[7:10]:
            return "#2A9D8F"
        else:
            return "#6C757D"

    wordcloud_custom = WordCloud(
        width=1200,
        height=800,
        background_color='white',
        max_words=100,
        relative_scaling=0.5
    ).generate_from_frequencies(immigration_dict)

    wordcloud_custom.recolor(color_func=color_func)

    plt.figure(figsize=(16, 10))
    plt.imshow(wordcloud_custom, interpolation='bilinear')
    plt.axis('off')
    plt.title('NHẬP CƯ VÀO CANADA - MÀU SẮC THEO THỨ HẠNG\n(Đỏ: Top 3 | Xanh dương: Top 4-7 | Xanh lá: Top 8-10 | Xám: Khác)',
              fontsize=14, pad=20, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ===================================================
    # THỐNG KÊ CHI TIẾT
    # ===================================================

    print("\n" + "="*60)
    print("THỐNG KÊ CHI TIẾT")
    print("="*60)

    print(f"\n📊 Tổng số người nhập cư (1980-2013): {total_immigration:,.0f}")
    print(f"🌍 Số quốc gia/vùng lãnh thổ: {len(df_filtered):,}")
    print(f"⭐ Trung bình mỗi quốc gia: {df_filtered['Total'].mean():,.0f}")
    print(f"📈 Quốc gia cao nhất: {df_filtered['Total'].idxmax()} ({df_filtered['Total'].max():,.0f})")
    print(f"📉 Quốc gia thấp nhất: {df_filtered['Total'].idxmin()} ({df_filtered['Total'].min():,.0f})")

    # Tính phân bố theo khu vực (nếu có cột Region)
    if 'Region' in df_filtered.columns:
        print("\n" + "="*60)
        print("PHÂN BỐ THEO KHU VỰC")
        print("="*60)

        # Kiểm tra kiểu dữ liệu của cột Region
        print(f"\nKiểu dữ liệu cột Region: {df_filtered['Region'].dtype}")

        # Chuyển đổi Region thành string nếu cần
        if df_filtered['Region'].dtype in ['int64', 'float64']:
            df_filtered['Region'] = df_filtered['Region'].astype(str)

        region_stats = df_filtered.groupby('Region')['Total'].sum().sort_values(ascending=False)

        for region, value in region_stats.items():
            pct = (value / total_immigration) * 100
            bar_length = int(pct / 2)
            bar = '█' * bar_length if bar_length > 0 else '░'
            region_name = str(region)[:35]  # Chuyển thành string và cắt ngắn
            print(f"{region_name:35s}: {value:>12,.0f} ({pct:>5.1f}%) {bar}")

    # Thống kê theo thập kỷ
    print("\n" + "="*60)
    print("PHÂN BỐ THEO THẬP KỶ")
    print("="*60)

    # Nhóm các năm theo thập kỷ
    decades = {
        '1980-1989': [col for col in year_columns if '198' in str(col)],
        '1990-1999': [col for col in year_columns if '199' in str(col)],
        '2000-2009': [col for col in year_columns if '200' in str(col)],
        '2010-2013': [col for col in year_columns if '201' in str(col)]
    }

    for decade, cols in decades.items():
        if cols:
            decade_total = df_filtered[cols].sum().sum()
            pct = (decade_total / total_immigration) * 100
            bar = '█' * int(pct / 2)
            print(f"{decade:15s}: {decade_total:>12,.0f} ({pct:>5.1f}%) {bar}")

else:
    print("Không tìm thấy dữ liệu phù hợp!")
    print(f"Số cột năm: {len(year_columns)}")
    print(f"Số dòng sau lọc: {len(df_filtered)}")

print("\n✅ HOÀN THÀNH!")